# Optimize Weld's Experiment with Ax


In [55]:
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig


In [ ]:
# Bootstrap MSVC toolchain inside the kernel process so BoTorch can build fused kernels on Windows.


def _load_msvc_env_from_vcvars() -> bool:
    vcvars = Path(
        r"C:\Program Files\Microsoft Visual Studio\18\Community\VC\Auxiliary\Build\vcvars64.bat"
    )
    if not vcvars.exists():
        print(f"MSVC bootstrap skipped: {vcvars} not found.")
        return False

    dump_env_cmd = Path(tempfile.gettempdir()) / "_dump_msvc_env.cmd"
    dump_env_cmd.write_text(
        f'@echo off\ncall "{vcvars}" >nul\nif errorlevel 1 exit /b 1\nset\n',
        encoding="utf-8",
    )

    try:
        proc = subprocess.run(
            ["cmd.exe", "/d", "/c", str(dump_env_cmd)],
            capture_output=True,
            text=True,
            encoding="mbcs",
            errors="replace",
            check=True,
        )
    except subprocess.CalledProcessError as exc:
        print(f"MSVC bootstrap failed with return code {exc.returncode}.")
        if exc.stderr:
            print(exc.stderr.strip())
        return False

    for line in proc.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value

    # Ensure the kernel keeps the active venv Scripts folder on PATH.
    scripts_dir = str(Path(sys.executable).resolve().parent)
    path_parts = os.environ.get("PATH", "").split(os.pathsep)
    if scripts_dir not in path_parts:
        os.environ["PATH"] = scripts_dir + os.pathsep + os.environ.get("PATH", "")

    return True


loaded = _load_msvc_env_from_vcvars()
print("MSVC env loaded:", loaded)
print("cl:", shutil.which("cl"))
print("ninja:", shutil.which("ninja"))

In [57]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig


In [58]:
NUM_NEW_EXP = 5

In [59]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [60]:
folder_path = Path(os.getcwd()) / "data"

In [61]:
exp = pd.read_excel(folder_path / "experiments.xlsx")
display(exp)


,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts
0,0,1700,70,20,round,0.105000,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1462,81,25,square,0.162470,O1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1733,44,18,round,0.102709,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1997,92,21,round,0.136314,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1649,63,13,square,0.055458,O1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
print(exp.columns)
cols_output = [
    "visual_quality",
    "uts",
]

cols_R1 = [
    "sigma_R1_M1",
    "sigma_R1_M2",
    "sigma_R1_M3",
    "sigma_R1_M4",
    "sigma_R1_M5",
]

cols_R2 = [
    "sigma_R2_M1",
    "sigma_R2_M2",
    "sigma_R2_M3",
    "sigma_R2_M4",
    "sigma_R2_M5",
]

cols_R3 = [
    "sigma_R3_M1",
    "sigma_R3_M2",
    "sigma_R3_M3",
    "sigma_R3_M4",
    "sigma_R3_M5",
]

Index(['trial_index', 'R', 'W', 'D', 'S', 'P', 'O', 'visual_quality',
       'sigma_R1_M1', 'sigma_R1_M2', 'sigma_R1_M3', 'sigma_R1_M4',
       'sigma_R1_M5', 'sigma_R2_M1', 'sigma_R2_M2', 'sigma_R2_M3',
       'sigma_R2_M4', 'sigma_R2_M5', 'sigma_R3_M1', 'sigma_R3_M2',
       'sigma_R3_M3', 'sigma_R3_M4', 'sigma_R3_M5', 'uts'],
      dtype='str')


In [63]:
filt_R1 = exp[cols_R1].isna().all(axis=1)
filt_R2 = exp[cols_R2].isna().all(axis=1)
filt_R3 = exp[cols_R3].isna().all(axis=1)
filt_output = exp[cols_output].isna().any(axis=1)
filt_removed = filt_R1 | filt_R2 | filt_R3 | filt_output
filt_removed

0    True
1    True
2    True
3    True
4    True
dtype: bool

In [64]:
exp_todo = exp[filt_removed].copy()
exp = exp[~filt_removed].copy()
display(exp)
display(exp_todo)

,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts


,trial_index,R,W,D,S,P,O,visual_quality,sigma_R1_M1,sigma_R1_M2,...,sigma_R2_M2,sigma_R2_M3,sigma_R2_M4,sigma_R2_M5,sigma_R3_M1,sigma_R3_M2,sigma_R3_M3,sigma_R3_M4,sigma_R3_M5,uts
0,0,1700,70,20,round,0.105000,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1462,81,25,square,0.162470,O1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1733,44,18,round,0.102709,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1997,92,21,round,0.136314,O2,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1649,63,13,square,0.055458,O1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 2. Configure where Ax will search.

CATEGORIES = {"S": ("square", "round"), "O": ("O1", "O2")}

bound_R = [1400, 2000]  # Rotatinal speed
bound_W = [40, 100]  # Welding speed
bound_D = [10, 30]  # Dwell time
bound_P = [0.03, 0.18]  # Plunge Depth

parameters = [
    RangeParameterConfig(
        name="R",
        bounds=bound_R,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="W",
        bounds=bound_W,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="D",
        bounds=bound_D,
        parameter_type="int",
    ),
    RangeParameterConfig(
        name="P",
        bounds=bound_P,
        parameter_type="float",
    ),
    ChoiceParameterConfig(
        name="S", parameter_type="str", values=list(CATEGORIES["S"]), is_ordered=True
    ),
    ChoiceParameterConfig(
        name="O", parameter_type="str", values=list(CATEGORIES["O"]), is_ordered=True
    ),
]

client.configure_experiment(
    name="parn_exp",
    parameters=parameters,
)

In [66]:
MAX_SIGMA_PEAK = 500
MIN_UTS = 200

# The framework will maximize the objective by default, so we use negative signs to minimize the values.
# The sigma_peak is to be minimized, while the uts is to be maximized. Therefore, we use -sigma_peak and uts in the objective.
client.configure_optimization(
    objective="-sigma_R1, -sigma_R2, -sigma_R3, uts",  # TODO: double check the sign
    outcome_constraints=[
        f"sigma_R1 <= {MAX_SIGMA_PEAK}",
        f"sigma_R2 <= {MAX_SIGMA_PEAK}",
        f"sigma_R3 <= {MAX_SIGMA_PEAK}",
        f"uts >= {MIN_UTS}",
        "visual_quality >= 0.5",
    ],
)

In [67]:
def cal_mean_sem(row, cols):
    filt = row[cols].notna()
    arr = row[cols][filt].astype(float).values

    if len(arr) <= 1:
        raise ValueError("Not enough data points to calculate mean and SEM.")

    mean = arr.mean()
    # NOTE: The ddof=1 because this is a sample standard deviation, not population.
    sem = arr.std(ddof=1) / np.sqrt(len(arr))
    return mean, sem

In [68]:
for idx, row in exp.iterrows():
    trial_index = client.attach_trial(
        # If Excel round-trips values as floats (e.g. 1499.9999999 due to float storage),
        # int() truncates down instead of rounding, silently shifting your attached parameter by 1.
        # Safer: int(round(row["R1"])).
        parameters={
            "R": int(round(row["R"])),
            "W": int(round(row["W"])),
            "D": int(round(row["D"])),
            "P": float(row["P"]),
            "S": row["S"],
            "O": row["O"],
        }
    )
    visual_quality = float(row["visual_quality"])
    uts = row["uts"]
    sigma_R1, sem_R1 = cal_mean_sem(row, cols_R1)
    sigma_R2, sem_R2 = cal_mean_sem(row, cols_R2)
    sigma_R3, sem_R3 = cal_mean_sem(row, cols_R3)

    print(
        f"Trial {trial_index}: visual_quality={visual_quality}, sigma_R1={sigma_R1}, sigma_R2={sigma_R2}, sigma_R3={sigma_R3}, uts={uts}"
    )
    client.complete_trial(
        trial_index=trial_index,
        raw_data={
            "visual_quality": visual_quality,
            "sigma_R1": (sigma_R1, sem_R1),
            "sigma_R2": (sigma_R2, sem_R2),
            "sigma_R3": (sigma_R3, sem_R3),
            "uts": uts,
        },
    )

In [69]:
if exp_todo.shape[0] == 0:
    suggest_params = []
    try:
        # Request only the intended number of new trials once.
        data = client.get_next_trials(max_trials=NUM_NEW_EXP)
        for trial_index, parameters in data.items():
            print(f"Trial {trial_index} with parameters {parameters}")
            suggest_params.append({"trial_index": trial_index, **parameters})
    except Exception as e:
        if type(e).__name__ == "DataRequiredError":
            print(
                "Ax cannot generate more trials yet: more observed trial data is "
                "required before moving past Sobol."
            )
        else:
            raise

    if suggest_params:
        _exp = pd.DataFrame(suggest_params)
        _exp[cols_output] = pd.NA
        exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
        display(exp)
        SAVE_EXCEL = True
    else:
        SAVE_EXCEL = False
else:
    print("There are still experiment to be conducted. Skip generating new sets")
    SAVE_EXCEL = False

There are still experiment to be conducted. Skip generating new sets


In [70]:
if SAVE_EXCEL:
    exp.to_excel(folder_path / "experiments.xlsx", index=False)